# LeetCode 904 — Fruit Into Baskets bằng Python

Link đề: https://leetcode.com/problems/fruit-into-baskets/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Sliding Window

Skeleton LeetCode:

```python
class Solution:
    def totalFruit(self, fruits: List[int]) -> int:
        ...
```

Ghi chú: đáp án duy nhất (một số nguyên). Điểm cốt lõi: bài thực chất là tìm **đoạn con liên tiếp dài nhất có tối đa 2 loại giá trị khác nhau** — **cửa sổ trượt** với bộ đếm. Oracle: **vét cạn mọi đoạn con** `[i, j]` và kiểm tra `len(set(...)) ≤ 2`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Dịch đề "hai giỏ, mỗi giỏ một loại, hái liên tiếp sang phải" thành bài **đoạn con dài nhất có tối đa 2 giá trị khác nhau**.
2. Nắm công cụ Python: `collections.Counter`/`dict` đếm, `len(dict)` là số loại, xoá khoá khi đếm về `0`, hai con trỏ `left`/`right`.
3. Đi từ **thử mọi điểm bắt đầu và mở rộng (`O(n²)`)** → **cửa sổ trượt co lại bằng `while`** → **cửa sổ chỉ trượt, không co** → **theo dõi hai loại gần nhất, `O(1)` bộ nhớ**.
4. Hiểu mấu chốt: khi thêm phần tử mới làm số loại vượt `2`, **thu hẹp từ trái** cho tới khi còn `≤ 2` loại; mỗi phần tử vào và ra cửa sổ tối đa một lần ⇒ `O(n)`.
5. Tránh lỗi kinh điển: không xoá khoá khi đếm về `0` (số loại bị đếm sai), thu hẹp chỉ **một bước** thay vì `while`, nhầm "hai loại phổ biến nhất" (không liên tiếp), cập nhật đáp án **trước** khi thu hẹp, và quên trường hợp mảng chỉ có một loại.

## 2. Đề bài & ví dụ

Một hàng cây ăn quả từ trái sang phải; `fruits[i]` là **loại** quả của cây thứ `i`.

Quy tắc:

- Bạn có **hai giỏ**, mỗi giỏ chỉ chứa **một loại** quả (không giới hạn số lượng).
- Bắt đầu từ **một cây bất kỳ**, bạn phải hái **đúng một quả** từ **mọi** cây (kể cả cây đầu) khi đi sang **phải**. Quả hái phải bỏ vừa vào một trong hai giỏ.
- Khi gặp cây có loại quả **không** vừa giỏ nào, bạn phải **dừng**.

Trả về **số quả tối đa** có thể hái.

**Ví dụ 1:** `fruits = [1,2,1]` → `3`.

**Ví dụ 2:** `fruits = [0,1,2,2]` → `3` (hái `[1,2,2]`).

**Ví dụ 3:** `fruits = [1,2,3,2,2]` → `4` (hái `[2,3,2,2]`).

**Ràng buộc:** `1 <= fruits.length <= 10^5`; `0 <= fruits[i] < fruits.length`.

In [1]:
import itertools
import random
import time
from collections import Counter, defaultdict
from typing import List

examples = [([1, 2, 1], 3), ([0, 1, 2, 2], 3), ([1, 2, 3, 2, 2], 4)]
for a, want in examples:
    print(f"fruits = {a} -> {want}")

fruits = [1, 2, 1] -> 3
fruits = [0, 1, 2, 2] -> 3
fruits = [1, 2, 3, 2, 2] -> 4


## 3. Kiến thức nền

### 3.1 Dịch đề: đoạn con liên tiếp có tối đa 2 loại

Hái liên tiếp từ cây `i` sang phải tới cây `j` thoả: các loại quả của cây `i..j` chỉ gồm tối đa **hai loại** (hai giỏ). Số quả hái được là `j − i + 1`. Vậy đáp án là độ dài lớn nhất của đoạn con có `≤ 2` loại.

In [2]:
a = [1, 2, 3, 2, 2]
print("đoạn [1..4] =", a[1:5], "-> số loại:", len(set(a[1:5])), "| độ dài:", 4)
print("đoạn [0..2] =", a[0:3], "-> số loại:", len(set(a[0:3])), "(3 loại: không hợp lệ)")

đoạn [1..4] = [2, 3, 2, 2] -> số loại: 2 | độ dài: 4
đoạn [0..2] = [1, 2, 3] -> số loại: 3 (3 loại: không hợp lệ)


### 3.2 Bộ đếm và số loại

`cnt = Counter()` đếm số quả mỗi loại trong cửa sổ. `len(cnt)` là số loại **hiện có** — chỉ đúng nếu ta **xoá khoá** khi đếm về `0` (hoặc đếm số khoá có giá trị `> 0`).

### 3.3 Cửa sổ trượt (sliding window)

Hai con trỏ `left ≤ right`; cửa sổ `fruits[left..right]`. Với mỗi `right`: thêm `fruits[right]`; **while** cửa sổ không hợp lệ (nhiều hơn 2 loại): bỏ `fruits[left]`, `left += 1`. Sau đó cửa sổ hợp lệ, cập nhật đáp án `right − left + 1`.

### 3.4 Vì sao `O(n)`

`right` chỉ tiến, `left` chỉ tiến, mỗi bước `while` tăng `left`: tổng bước `≤ 2n`.

In [3]:
cnt = Counter()
for x in [1, 2, 3]:
    cnt[x] += 1
print(dict(cnt), "số loại =", len(cnt))
cnt[1] -= 1
print("giảm 1 về 0 nhưng chưa xoá khoá:", dict(cnt), "len =", len(cnt), "(sai: chỉ còn 2 loại)")
del cnt[1]
print("sau khi xoá khoá:", dict(cnt), "len =", len(cnt))

{1: 1, 2: 1, 3: 1} số loại = 3
giảm 1 về 0 nhưng chưa xoá khoá: {1: 0, 2: 1, 3: 1} len = 3 (sai: chỉ còn 2 loại)
sau khi xoá khoá: {2: 1, 3: 1} len = 2


## 4. Ý tưởng nền tảng / trực giác

`fruits = [1, 2, 3, 2, 2]`. Trượt cửa sổ:

| `right` | thêm | cửa sổ `[left..right]` | số loại | thu hẹp | độ dài |
|---|---|---|---|---|---|
| 0 | `1` | `[1]` | 1 | — | 1 |
| 1 | `2` | `[1,2]` | 2 | — | 2 |
| 2 | `3` | `[1,2,3]` | 3 > 2 | bỏ `1` (`left = 1`) | `[2,3]`: 2 |
| 3 | `2` | `[2,3,2]` | 2 | — | 3 |
| 4 | `2` | `[2,3,2,2]` | 2 | — | **4** |

Đáp án `4`.

In [4]:
fruits = [1, 2, 3, 2, 2]
cnt, left, best = Counter(), 0, 0
for right, x in enumerate(fruits):
    cnt[x] += 1
    while len(cnt) > 2:
        y = fruits[left]
        cnt[y] -= 1
        if cnt[y] == 0:
            del cnt[y]
        left += 1
    best = max(best, right - left + 1)
    print(f"right={right} thêm {x}: cửa sổ {fruits[left:right + 1]} loại={len(cnt)} -> độ dài {right - left + 1}, best={best}")

right=0 thêm 1: cửa sổ [1] loại=1 -> độ dài 1, best=1
right=1 thêm 2: cửa sổ [1, 2] loại=2 -> độ dài 2, best=2
right=2 thêm 3: cửa sổ [2, 3] loại=2 -> độ dài 2, best=2
right=3 thêm 2: cửa sổ [2, 3, 2] loại=2 -> độ dài 3, best=3
right=4 thêm 2: cửa sổ [2, 3, 2, 2] loại=2 -> độ dài 4, best=4


## 5. Cách 1 — Brute Force: thử mọi điểm bắt đầu rồi mở rộng

**Ý tưởng:** với mỗi cây bắt đầu `i`, đi sang phải, dùng `set` các loại đã gặp; dừng khi loại thứ `3` xuất hiện. Độ dài đoạn hái được là `j − i`; lấy cực đại.

In [5]:
def fruit_brute(fruits):
    n, best = len(fruits), 0
    for i in range(n):
        types = set()
        j = i
        while j < n:
            types.add(fruits[j])
            if len(types) > 2:                           # loại thứ 3: phải dừng trước cây j
                break
            j += 1
        best = max(best, j - i)                          # đã hái cây i..j-1
    return best


for a, want in examples:
    assert fruit_brute(a) == want
print([fruit_brute(a) for a, _ in examples])

[3, 3, 4]


In [6]:
# Code nộp LeetCode cho Cách 1 (đúng nhưng O(n^2): TLE khi n = 10^5)
class SolutionBrute:
    def totalFruit(self, fruits: List[int]) -> int:
        n, best = len(fruits), 0
        for i in range(n):
            types = set()
            j = i
            while j < n:
                types.add(fruits[j])
                if len(types) > 2:
                    break
                j += 1
            best = max(best, j - i)
        return best

**Độ phức tạp:** `O(n²)` (đoạn dài với ít loại, như mảng toàn một loại, làm mỗi điểm đầu quét tới cuối); bộ nhớ `O(1)`.

**Nút thắt:** khi chuyển điểm bắt đầu từ `i` sang `i + 1`, ta quét lại gần như toàn bộ phần đã quét. Cửa sổ có thể **tái sử dụng**.

## 6. Ý tưởng thuật toán / bước đột phá

Duy trì cửa sổ `[left, right]` luôn hợp lệ (`≤ 2` loại): đẩy `right`; nếu thành `3` loại thì đẩy `left` tới khi còn `2`. Điểm bắt đầu tốt nhất cho điểm cuối `right` là `left` nhỏ nhất hợp lệ, và `left` **không bao giờ giảm** khi `right` tăng (nếu `[l, r]` hợp lệ thì mọi `[l', r]` với `l' ≥ l` cũng hợp lệ; ngược lại `[l, r+1]` không hợp lệ thì mọi `l'' ≤ l` cũng không). Nhờ đó mỗi con trỏ chỉ tiến ⇒ `O(n)`.

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** cửa sổ trượt co bằng `while`, **Cách 3** cửa sổ chỉ trượt (không co), **Cách 4** theo dõi hai loại gần nhất (`O(1)` bộ nhớ).

### Cách 2 — Cửa sổ trượt co bằng `while`

**Giải thích từng đoạn:**

1. `cnt` đếm số quả mỗi loại trong cửa sổ; `left = 0`.
2. Với mỗi `right`: `cnt[fruits[right]] += 1`.
3. `while len(cnt) > 2`: giảm `cnt[fruits[left]]`, **xoá khoá nếu về `0`**, `left += 1`.
4. `best = max(best, right − left + 1)`.

In [7]:
def fruit_window(fruits):
    cnt, left, best = {}, 0, 0
    for right, x in enumerate(fruits):
        cnt[x] = cnt.get(x, 0) + 1
        while len(cnt) > 2:                              # quá 2 loại: co từ trái
            y = fruits[left]
            cnt[y] -= 1
            if cnt[y] == 0:
                del cnt[y]                               # nhớ xoá khoá để len(cnt) đúng
            left += 1
        best = max(best, right - left + 1)
    return best


for a, want in examples:
    assert fruit_window(a) == want
print([fruit_window(a) for a, _ in examples])

[3, 3, 4]


In [8]:
# Code nộp LeetCode cho Cách 2
class SolutionWindow:
    def totalFruit(self, fruits: List[int]) -> int:
        cnt, left, best = {}, 0, 0
        for right, x in enumerate(fruits):
            cnt[x] = cnt.get(x, 0) + 1
            while len(cnt) > 2:
                y = fruits[left]
                cnt[y] -= 1
                if cnt[y] == 0:
                    del cnt[y]
                left += 1
            best = max(best, right - left + 1)
        return best

**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ (tối đa 3 khoá trong `cnt`).

### Cách 3 — Cửa sổ chỉ trượt, không co

**Ý tưởng:** ta chỉ cần **độ dài lớn nhất**, không cần cửa sổ đúng ở mọi lúc. Giữ độ dài cửa sổ **không giảm**: mỗi `right` mới đẩy `right`, và nếu cửa sổ không hợp lệ thì chỉ đẩy `left` **một bước** (cửa sổ "trượt" giữ nguyên độ dài). Cuối cùng độ dài cửa sổ `= n − left` là đáp án (độ dài chỉ tăng khi tìm được cửa sổ hợp lệ dài hơn).

In [9]:
def fruit_slide(fruits):
    cnt, left = {}, 0
    for right, x in enumerate(fruits):
        cnt[x] = cnt.get(x, 0) + 1
        if len(cnt) > 2:                                 # không hợp lệ: chỉ trượt trái đúng một bước (độ dài không giảm)
            y = fruits[left]
            cnt[y] -= 1
            if cnt[y] == 0:
                del cnt[y]
            left += 1
    return len(fruits) - left                            # độ dài cửa sổ cuối = độ dài tốt nhất từng đạt


for a, want in examples:
    assert fruit_slide(a) == want
print([fruit_slide(a) for a, _ in examples])

[3, 3, 4]


In [10]:
# Code nộp LeetCode cho Cách 3
class SolutionSlide:
    def totalFruit(self, fruits: List[int]) -> int:
        cnt, left = {}, 0
        for x in fruits:
            cnt[x] = cnt.get(x, 0) + 1
            if len(cnt) > 2:
                y = fruits[left]
                cnt[y] -= 1
                if cnt[y] == 0:
                    del cnt[y]
                left += 1
        return len(fruits) - left

**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ, mã gọn hơn (không `while`). Cửa sổ có thể chứa 3 loại tại một số thời điểm, nhưng **độ dài** của nó vẫn bằng độ dài lớn nhất của một đoạn hợp lệ đã từng gặp (chứng minh ở mục 10(d)).

### Cách 4 — Theo dõi hai loại gần nhất

**Ý tưởng:** duyệt một lượt, giữ: `a`, `b` là hai loại của đoạn hợp lệ dài nhất kết thúc tại vị trí hiện tại (`b` là loại **vừa gặp gần nhất**), `run_b` = độ dài dãy liên tiếp cuối cùng của loại `b`, `cur` = độ dài đoạn hợp lệ hiện tại.

- Gặp `x ∈ {a, b}`: `cur += 1`. Nếu `x == b`: `run_b += 1`; ngược lại `a, b = b, x`, `run_b = 1`.
- Gặp loại mới `x`: đoạn hợp lệ mới bắt đầu từ dãy cuối `b`: `cur = run_b + 1`; `a, b = b, x`; `run_b = 1`.
- `best = max(best, cur)`.

In [11]:
def fruit_two_last(fruits):
    a = b = -1                                           # a: loại cũ hơn, b: loại gần nhất
    run_b = cur = best = 0                               # run_b: độ dài dãy liên tiếp cuối cùng của loại b
    for x in fruits:
        if x == a or x == b:
            cur += 1                                     # vẫn trong hai loại
        else:
            cur = run_b + 1                              # loại mới: giữ lại đúng dãy cuối của b
        if x == b:
            run_b += 1
        else:
            run_b = 1
            a, b = b, x                                  # x trở thành loại gần nhất
        best = max(best, cur)
    return best


for a, want in examples:
    assert fruit_two_last(a) == want
print([fruit_two_last(a) for a, _ in examples])

[3, 3, 4]


In [12]:
# Code nộp LeetCode cho Cách 4
class SolutionTwoLast:
    def totalFruit(self, fruits: List[int]) -> int:
        a = b = -1
        run_b = cur = best = 0
        for x in fruits:
            if x == a or x == b:
                cur += 1
            else:
                cur = run_b + 1
            if x == b:
                run_b += 1
            else:
                run_b = 1
                a, b = b, x
            best = max(best, cur)
        return best

**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ, không dùng `dict`.

## 8. TRACE từng bước

**Trace Cách 2** với `fruits = [1,2,3,2,2]` (đã in ở mục 4). **Trace Cách 4** với cùng mảng:

| `x` | `(a, b)` trước | `cur` | `run_b` | `(a, b)` sau | `best` |
|---|---|---|---|---|---|
| 1 | `(-1,-1)` | `0 + 1 = 1` (loại mới, `run_b = 0`) | 1 | `(-1, 1)` | 1 |
| 2 | `(-1, 1)` | loại mới: `run_b + 1 = 2` | 1 | `(1, 2)` | 2 |
| 3 | `(1, 2)` | loại mới: `run_b + 1 = 2` | 1 | `(2, 3)` | 2 |
| 2 | `(2, 3)` | thuộc `{2,3}`: `cur = 3` | 1 (loại `2` không phải `b`) | `(3, 2)` | 3 |
| 2 | `(3, 2)` | `x == b`: `cur = 4` | 2 | `(3, 2)` | **4** |

Hàm trace:

In [13]:
def trace_two_last(fruits):
    a = b = -1
    run_b = cur = best = 0
    print(f"{'x':>2} {'(a,b) trước':>12} {'cur':>4} {'run_b':>6} {'(a,b) sau':>11} {'best':>5}")
    for x in fruits:
        before = (a, b)
        cur = cur + 1 if (x == a or x == b) else run_b + 1
        if x == b:
            run_b += 1
        else:
            run_b = 1
            a, b = b, x
        best = max(best, cur)
        print(f"{x:>2} {str(before):>12} {cur:>4} {run_b:>6} {str((a, b)):>11} {best:>5}")
    return best


assert trace_two_last([1, 2, 3, 2, 2]) == 4

 x  (a,b) trước  cur  run_b   (a,b) sau  best
 1     (-1, -1)    1      1     (-1, 1)     1
 2      (-1, 1)    2      1      (1, 2)     2
 3       (1, 2)    2      1      (2, 3)     2
 2       (2, 3)    3      1      (3, 2)     3
 2       (3, 2)    4      2      (3, 2)     4


## 9. Code nộp LeetCode

Chọn **Cách 2** (cửa sổ trượt co bằng `while`, rõ ràng và tổng quát). Giữ nguyên `class Solution`, `totalFruit(self, fruits)`:

In [14]:
class Solution:
    def totalFruit(self, fruits: List[int]) -> int:
        cnt = {}                                          # loại quả -> số quả trong cửa sổ
        left = best = 0
        for right, x in enumerate(fruits):
            cnt[x] = cnt.get(x, 0) + 1
            while len(cnt) > 2:                           # quá hai loại: co cửa sổ từ trái
                y = fruits[left]
                cnt[y] -= 1
                if cnt[y] == 0:
                    del cnt[y]                            # xoá khoá để len(cnt) là số loại thật
                left += 1
            best = max(best, right - left + 1)
        return best


print([Solution().totalFruit(a) for a, _ in examples])      # [3, 3, 4]
assert all(Solution().totalFruit(a) == want for a, want in examples)

[3, 3, 4]


## 10. Vì sao đúng

**(a) Quy về đoạn con.** Hái liên tiếp từ `i` tới `j` hợp lệ ⇔ các loại trong `fruits[i..j]` chiếm tối đa 2 giá trị (mỗi giỏ một loại, hái đủ mọi cây trên đường). Số quả `j − i + 1`. Tối đa hoá ⇒ đoạn con dài nhất có `≤ 2` loại.

**(b) Tính đơn điệu của `left`.** Với `r` cố định gọi `L(r)` là điểm đầu nhỏ nhất sao cho `[L(r), r]` có `≤ 2` loại. Nếu `[l, r+1]` hợp lệ thì `[l, r]` cũng hợp lệ (đoạn con của đoạn hợp lệ), suy ra `L(r) ≤ L(r + 1)`. Vậy `L` không giảm.

**(c) Cách 2.** Sau khi xử lý `right`, vòng `while` đẩy `left` tới đúng `L(right)` (nó dừng khi lần đầu còn `≤ 2` loại, và nhờ (b) không vượt qua `L`). Đáp án `max_r (r − L(r) + 1)` chính là độ dài lớn nhất.

**(d) Cách 3 (không co).** Gọi `w_r = r + 1 − left_r` độ dài cửa sổ sau khi xử lý `right = r`. Khi thêm một phần tử, độ dài tăng `1`; nếu bị vi phạm ta đẩy `left` một bước thì độ dài trở về `w_{r−1}`. Vậy `w_r ∈ {w_{r−1}, w_{r−1} + 1}`: không giảm. Khi `w_r = w_{r−1} + 1` thì cửa sổ **không vi phạm** (không phải đẩy `left`) nên đó là đoạn hợp lệ thật dài `w_r`. Vậy `w_r` bằng độ dài lớn nhất của đoạn hợp lệ kết thúc tại một `r' ≤ r`; thật vậy bằng quy nạp: `w_r = max_{r' ≤ r}` (độ dài lớn nhất hợp lệ). Cuối cùng `w_{n−1} = n − left` là đáp án. (Cửa sổ ở thời điểm vi phạm có thể chứa 3 loại nhưng độ dài không vượt đáp án đã đạt.)

**(e) Cách 4.** Với điểm cuối `i`, đoạn hợp lệ dài nhất kết thúc tại `i` chứa hai loại: loại `x_i` và loại xuất hiện ngay trước dãy cuối của `x_i`. Ký hiệu `b` là loại gần nhất và `run_b` độ dài dãy liên tiếp cuối cùng của `b`. Nếu loại mới `x ∉ {a, b}` thì đoạn hợp lệ dài nhất kết thúc tại `x` bắt đầu ngay tại đầu dãy cuối của `b`: độ dài `run_b + 1`. Nếu `x ∈ {a, b}`: nối vào đoạn hiện tại: `cur + 1`. Cập nhật `(a, b)` và `run_b` giữ bất biến.

Ta kiểm chứng bằng máy: mọi cách khớp oracle **vét cạn mọi đoạn con** trên mọi mảng nhỏ (mục 12D) và khớp nhau trên mảng lớn; ngoài ra Cách 3 được kiểm tra `w_r` không giảm (12E).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Thử mọi điểm bắt đầu | `O(n²)` | `O(1)` | oracle một phần; TLE |
| 2. Cửa sổ co bằng `while` | `O(n)` | `O(1)` | **đáp án chính** |
| 3. Cửa sổ chỉ trượt | `O(n)` | `O(1)` | mã gọn, không `while` |
| 4. Hai loại gần nhất | `O(n)` | `O(1)` | không `dict` |

Đo thực tế (`n = 10^5`; cách 1 chỉ chạy dữ liệu ngẫu nhiên và chuỗi 2 loại ngắn):

In [15]:
def bench(fn, arg, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(arg)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(904)
n = 10 ** 5
datasets = {
    "ngẫu nhiên nhiều loại": [rng.randrange(n) for _ in range(n)],
    "ngẫu nhiên 3 loại": [rng.randrange(3) for _ in range(n)],
    "toàn một loại": [7] * n,
}
for label, data in datasets.items():
    row = [f"{name} {bench(fn, data) * 1e3:8.2f} ms" for name, fn in
           [("window", fruit_window), ("slide", fruit_slide), ("two_last", fruit_two_last)]]
    print(f"{label:22s}: " + " | ".join(row))
print(f"cách 1 với n = 5000 toàn một loại: {bench(fruit_brute, [7] * 5000, 1) * 1e3:8.1f} ms (tăng theo n²)")

ngẫu nhiên nhiều loại : window    26.11 ms | slide    20.61 ms | two_last     7.53 ms
ngẫu nhiên 3 loại     : window    22.66 ms | slide    14.19 ms | two_last     7.85 ms


toàn một loại         : window    13.15 ms | slide     7.98 ms | two_last     9.23 ms


cách 1 với n = 5000 toàn một loại:   1023.2 ms (tăng theo n²)


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi mảng nhỏ với oracle mọi-đoạn-con, (E) ngẫu nhiên có seed + tính chất, (F) giới hạn.

### A. Ví dụ của đề

In [16]:
def oracle(fruits):
    """Đúng định nghĩa: đoạn con [i, j] hợp lệ nếu len(set(fruits[i..j])) <= 2; đáp án = độ dài lớn nhất."""
    n = len(fruits)
    return max(j - i + 1 for i in range(n) for j in range(i, n) if len(set(fruits[i:j + 1])) <= 2)


IMPLS = {
    "brute": fruit_brute,
    "window": fruit_window,
    "slide": fruit_slide,
    "two_last": fruit_two_last,
    "Solution": lambda a: Solution().totalFruit(list(a)),
    "SolutionBrute": lambda a: SolutionBrute().totalFruit(list(a)),
    "SolutionWindow": lambda a: SolutionWindow().totalFruit(list(a)),
    "SolutionSlide": lambda a: SolutionSlide().totalFruit(list(a)),
    "SolutionTwoLast": lambda a: SolutionTwoLast().totalFruit(list(a)),
}


def check(fruits, want=None, names=None):
    if want is None:
        want = oracle(fruits)
    for name in (names or IMPLS):
        got = IMPLS[name](list(fruits))
        assert got == want, (name, fruits, got, want)


for a, want in examples:
    assert oracle(a) == want
    check(a, want)
print("ví dụ đề: oracle mọi-đoạn-con và", len(IMPLS), "cài đặt đúng")

ví dụ đề: oracle mọi-đoạn-con và 9 cài đặt đúng


### B. Edge case

- Một cây.
- Toàn một loại: đáp án `n`.
- Đúng hai loại: đáp án `n`.
- Ba loại xen kẽ `[0,1,2,0,1,2]`: `2`.
- Đoạn tốt nhất ở **cuối** mảng.
- Đoạn tốt nhất ở **đầu** mảng.
- Loại quay lại sau một khoảng: `[1,2,1,3,1]`.
- Quay lại loại cũ sau khi đổi: `[1,2,2,3,3,2]`.

In [17]:
cases = [
    ([0], 1), ([5, 5, 5, 5], 4), ([1, 2, 1, 2, 1], 5), ([0, 1, 2, 0, 1, 2], 2),
    ([3, 1, 1, 2, 2, 2], 5), ([1, 1, 2, 2, 3, 4], 4), ([1, 2, 1, 3, 1], 3), ([1, 2, 2, 3, 3, 2], 5),
    ([0, 1], 2), ([0, 1, 2], 2), ([4, 4, 3, 3, 3, 2, 2, 2, 2], 7), ([1, 0, 1, 4, 1, 4, 1, 2, 3], 5),
]
for a, want in cases:
    check(a, want)
print(len(cases), "edge case: đúng")

12 edge case: đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [18]:
def bug_no_delete(fruits):                             # không xoá khoá khi đếm về 0
    cnt, left, best = {}, 0, 0
    for right, x in enumerate(fruits):
        cnt[x] = cnt.get(x, 0) + 1
        while len(cnt) > 2:
            y = fruits[left]
            cnt[y] -= 1                                # SAI: khoá vẫn còn với giá trị 0
            left += 1
            if left > right:
                break
        best = max(best, right - left + 1)
    return best


def bug_keep_only_previous(fruits):                    # gặp loại thứ ba thì chỉ giữ MỘT cây ngay trước (không giữ cả dãy cuối)
    best, left, types = 0, 0, set()
    for right, x in enumerate(fruits):
        if x not in types and len(types) == 2:
            left = right - 1                           # SAI: dãy cuối của loại gần nhất có thể dài hơn 1
            types = {fruits[right - 1]}
        types.add(x)
        best = max(best, right - left + 1)
    return best


def bug_two_most_frequent(fruits):                     # đếm quả của hai loại phổ biến nhất (không liên tiếp)
    c = Counter(fruits).most_common(2)
    return sum(v for _, v in c)


def bug_update_before_shrink(fruits):                  # cập nhật đáp án trước khi co
    cnt, left, best = {}, 0, 0
    for right, x in enumerate(fruits):
        cnt[x] = cnt.get(x, 0) + 1
        best = max(best, right - left + 1)             # SAI: cửa sổ có thể đang có 3 loại
        while len(cnt) > 2:
            y = fruits[left]
            cnt[y] -= 1
            if cnt[y] == 0:
                del cnt[y]
            left += 1
    return best


def bug_reset_on_third(fruits):                        # gặp loại thứ ba thì bắt đầu lại từ vị trí hiện tại (bỏ dãy cuối của loại gần nhất)
    best, start, types = 0, 0, set()
    for i, x in enumerate(fruits):
        if x not in types and len(types) == 2:
            start, types = i, set()
        types.add(x)
        best = max(best, i - start + 1)
    return best


def bug_len_minus_one(fruits):                         # độ dài tính thiếu 1 (right - left)
    cnt, left, best = {}, 0, 0
    for right, x in enumerate(fruits):
        cnt[x] = cnt.get(x, 0) + 1
        while len(cnt) > 2:
            y = fruits[left]
            cnt[y] -= 1
            if cnt[y] == 0:
                del cnt[y]
            left += 1
        best = max(best, right - left)
    return best


BUGS = [bug_no_delete, bug_keep_only_previous, bug_two_most_frequent, bug_update_before_shrink, bug_reset_on_third, bug_len_minus_one]
rng = random.Random(3)
tests = [[rng.randint(0, 3) for _ in range(rng.randint(1, 9))] for _ in range(800)]
for bug in BUGS:
    bad = next((a for a in tests if bug(a) != oracle(a)), None)
    assert bad is not None, bug.__name__
    print(f"{bug.__name__:25s} sai trên {bad}: ra {bug(bad)}, đúng {oracle(bad)}")

bug_no_delete             sai trên [2, 0, 0, 3, 3]: ra 3, đúng 4
bug_keep_only_previous    sai trên [2, 0, 0, 3, 3]: ra 3, đúng 4
bug_two_most_frequent     sai trên [2, 3, 3, 2, 3, 1, 2]: ra 6, đúng 5
bug_update_before_shrink  sai trên [1, 2, 3, 0]: ra 3, đúng 2
bug_reset_on_third        sai trên [2, 0, 0, 3, 3]: ra 3, đúng 4
bug_len_minus_one         sai trên [1, 2, 3, 0]: ra 1, đúng 2


### D. Vét cạn mọi mảng nhỏ

Mọi mảng độ dài `1..8` với giá trị `0..3` (`4 + 16 + 64 + 256 + 1 024 + 4 096 + 16 384 + 65 536 = 87 380`) — mọi dạng, kể cả mảng có `3` và `4` loại. Oracle: mọi đoạn con.

In [19]:
t0 = time.perf_counter()
cnt = 0
fast = ("brute", "window", "slide", "two_last", "Solution")
for n in range(1, 9):
    for tup in itertools.product(range(4), repeat=n):
        a = list(tup)
        want = oracle(a)
        for name in fast:
            assert IMPLS[name](a) == want, (name, a)
        cnt += 1
print(cnt, "mảng nhỏ: 5 cài đặt khớp oracle mọi-đoạn-con;", round(time.perf_counter() - t0, 1), "s")
assert cnt == 87380

87380 mảng nhỏ: 5 cài đặt khớp oracle mọi-đoạn-con; 1.9 s


### E. Ngẫu nhiên có seed và tính chất

Mảng ngẫu nhiên tới `60` phần tử, số loại `1..6`. Tính chất: (1) đáp án `≥` độ dài dãy liên tiếp dài nhất của một loại và `≤ n`; (2) đảo mảng không đổi đáp án; (3) gán lại nhãn loại (đổi tên giá trị) không đổi đáp án; (4) độ dài cửa sổ của Cách 3 **không giảm** theo `right`.

In [20]:
rng = random.Random(20240904)
for _ in range(2000):
    n = rng.randint(1, 60)
    k = rng.randint(1, 6)
    a = [rng.randrange(k) for _ in range(n)]
    want = oracle(a) if n <= 30 else fruit_window(a)
    for name in ("window", "slide", "two_last", "Solution", "SolutionWindow", "SolutionSlide", "SolutionTwoLast"):
        assert IMPLS[name](a) == want, (name, a)
    longest_run = max(len(list(g)) for _, g in itertools.groupby(a))
    assert longest_run <= want <= n                                                  # (1)
    assert Solution().totalFruit(a[::-1]) == want                                    # (2)
    perm = list(range(k))
    rng.shuffle(perm)
    assert Solution().totalFruit([perm[x] for x in a]) == want                       # (3)
    cnt, left, widths = {}, 0, []
    for right, x in enumerate(a):                                                    # (4) cửa sổ của cách 3
        cnt[x] = cnt.get(x, 0) + 1
        if len(cnt) > 2:
            y = a[left]
            cnt[y] -= 1
            if cnt[y] == 0:
                del cnt[y]
            left += 1
        widths.append(right + 1 - left)
    assert all(widths[i] <= widths[i + 1] for i in range(len(widths) - 1)) and widths[-1] == want
print("2000 mảng ngẫu nhiên: 7 cài đặt khớp; 4 tính chất đúng (cận, đảo, đổi nhãn, độ rộng cửa sổ không giảm)")

2000 mảng ngẫu nhiên: 7 cài đặt khớp; 4 tính chất đúng (cận, đảo, đổi nhãn, độ rộng cửa sổ không giảm)


### F. Giới hạn: `n = 10^5`

Các cách `O(n)` khớp nhau trên mảng lớn nhiều dạng; công thức đóng cho dạng có thể tính tay.

In [21]:
rng = random.Random(500)
n = 10 ** 5
fast = ["window", "slide", "two_last", "Solution", "SolutionWindow", "SolutionSlide", "SolutionTwoLast"]
datasets = {
    "ngẫu nhiên nhiều loại": [rng.randrange(n) for _ in range(n)],
    "ngẫu nhiên 2 loại": [rng.randrange(2) for _ in range(n)],
    "ngẫu nhiên 3 loại": [rng.randrange(3) for _ in range(n)],
    "toàn một loại": [7] * n,
    "mọi loại khác nhau": list(range(n)),
    "xen kẽ 0 1 0 1": [i % 2 for i in range(n)],
    "ba loại xen kẽ": [i % 3 for i in range(n)],
    "khối 2 loại rồi khối khác": [0] * 30000 + [1] * 30000 + [2] * 30000 + [3] * 10000,
}
for name, a in datasets.items():
    res = {k: IMPLS[k](a) for k in fast}
    assert len(set(res.values())) == 1, (name, res)
    print(f"{name:26s}: đáp án {res['window']}")
assert Solution().totalFruit(datasets["toàn một loại"]) == n
assert Solution().totalFruit(datasets["mọi loại khác nhau"]) == 2
assert Solution().totalFruit(datasets["xen kẽ 0 1 0 1"]) == n
assert Solution().totalFruit(datasets["ba loại xen kẽ"]) == 2
assert Solution().totalFruit(datasets["khối 2 loại rồi khối khác"]) == 60000
print("công thức đóng khớp")

ngẫu nhiên nhiều loại     : đáp án 3
ngẫu nhiên 2 loại         : đáp án 100000


ngẫu nhiên 3 loại         : đáp án 31
toàn một loại             : đáp án 100000


mọi loại khác nhau        : đáp án 2
xen kẽ 0 1 0 1            : đáp án 100000


ba loại xen kẽ            : đáp án 2
khối 2 loại rồi khối khác : đáp án 60000


công thức đóng khớp


## 13. Các lỗi người mới hay gặp

### Lỗi 1: không xoá khoá khi đếm về `0`

`len(cnt)` vẫn tính loại đã hết quả: cửa sổ co quá tay hoặc không bao giờ dừng đúng.

In [22]:
bad = next(a for a in tests if bug_no_delete(a) != oracle(a))
print(bad, "->", bug_no_delete(bad), "| đúng:", oracle(bad))
cnt = Counter([1, 2, 3])
cnt[1] -= 1
print("Counter sau khi giảm 1 về 0:", dict(cnt), "len =", len(cnt), "(vẫn là 3, không phải 2)")
assert len(cnt) == 3

[2, 0, 0, 3, 3] -> 3 | đúng: 4
Counter sau khi giảm 1 về 0: {1: 0, 2: 1, 3: 1} len = 3 (vẫn là 3, không phải 2)


**Vì sao sai:** số loại thật là số khoá có giá trị `> 0`. **Đúng:** `if cnt[y] == 0: del cnt[y]`.

### Lỗi 2: chỉ giữ một cây ngay trước khi gặp loại thứ ba

In [23]:
bad = next(a for a in tests if bug_keep_only_previous(a) != oracle(a))
print(bad, "->", bug_keep_only_previous(bad), "| đúng:", oracle(bad))

[2, 0, 0, 3, 3] -> 3 | đúng: 4


**Vì sao sai:** khi gặp loại thứ ba, đoạn mới phải giữ **cả dãy liên tiếp cuối cùng** của loại ngay trước (có thể nhiều cây), không chỉ một cây. **Đúng:** co từ trái bằng `while` tới khi còn 2 loại (hoặc theo dõi `run_b`). (Lưu ý: nếu chỉ co **một bước** nhưng đáp án đọc từ độ dài cửa sổ không giảm thì lại **đúng** — đó chính là Cách 3.)

### Lỗi 3: lấy hai loại phổ biến nhất

In [24]:
print("[1,2,1,3,3,3,1]:", bug_two_most_frequent([1, 2, 1, 3, 3, 3, 1]), "| đúng:", oracle([1, 2, 1, 3, 3, 3, 1]))
assert bug_two_most_frequent([1, 2, 1, 3, 3, 3, 1]) == 6 and oracle([1, 2, 1, 3, 3, 3, 1]) == 5

[1,2,1,3,3,3,1]: 6 | đúng: 5


**Vì sao sai:** phải hái **liên tiếp**, không thể nhảy cóc. **Đúng:** đoạn con liên tiếp.

### Lỗi 4: cập nhật đáp án trước khi co

In [25]:
bad = next(a for a in tests if bug_update_before_shrink(a) != oracle(a))
print(bad, "->", bug_update_before_shrink(bad), "| đúng:", oracle(bad))

[1, 2, 3, 0] -> 3 | đúng: 2


**Vì sao sai:** ngay sau khi thêm, cửa sổ có thể chứa 3 loại (không hợp lệ). **Đúng:** co xong rồi mới cập nhật `best`.

### Lỗi 5: reset hoàn toàn khi gặp loại thứ ba

In [26]:
print("[1,2,2,3,3,2]:", bug_reset_on_third([1, 2, 2, 3, 3, 2]), "| đúng:", oracle([1, 2, 2, 3, 3, 2]))
assert bug_reset_on_third([1, 2, 2, 3, 3, 2]) != oracle([1, 2, 2, 3, 3, 2])

[1,2,2,3,3,2]: 3 | đúng: 5


**Vì sao sai:** khi gặp loại thứ ba, đoạn tốt nhất tiếp theo vẫn giữ **dãy cuối của loại gần nhất** (ví dụ `2,3,3,2`). Reset từ vị trí hiện tại bỏ mất. **Đúng:** co từ trái tới khi còn 2 loại, hoặc giữ `run_b`.

### Lỗi 6 (bonus): độ dài thiếu 1

In [27]:
print("[5]:", bug_len_minus_one([5]), "| đúng:", Solution().totalFruit([5]))
assert bug_len_minus_one([5]) == 0

[5]: 0 | đúng: 1


**Vì sao sai:** cửa sổ `[left, right]` có `right − left + 1` phần tử. **Đúng:** `+ 1`.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "đoạn con liên tiếp dài nhất với giới hạn số **loại khác nhau**" (hoặc tổng/số lần lặp) → **cửa sổ trượt** với bộ đếm; điều kiện hợp lệ **đơn điệu** (bỏ bớt phần tử không làm hỏng).

**Mẹo:**

1. Dịch đề sang ngôn ngữ đoạn con trước.
2. Bộ đếm + **xoá khoá** khi về `0`.
3. `while` để co đến khi hợp lệ; đáp án cập nhật **sau** khi co.
4. Mẹo cửa sổ không co: chỉ cần độ dài lớn nhất nên độ dài cửa sổ không giảm.
5. Khi số loại cố định rất nhỏ (2), có thể dùng biến thay `dict`.
6. Oracle: mọi đoạn con.

**Họ bài cùng pattern:** Longest Substring with At Most K Distinct Characters (340), Longest Substring Without Repeating Characters (3), Max Consecutive Ones III (1004), Longest Repeating Character Replacement (424), Subarrays with K Different Integers (992), Fruit Into Baskets II (3477, khác).

## 15. Biến thể

1. **Tối đa `k` loại** (340/904 tổng quát).
2. **Số đoạn con có tối đa 2 loại** (đếm).
3. **Đúng 2 loại** (hơn: Subarrays with K Different Integers — hiệu của "tối đa 2" và "tối đa 1").
4. **Trả về chính đoạn** (chỉ số bắt đầu/kết thúc).
5. **Số lượng quả mỗi loại có trọng số** (giỏ có sức chứa).

Cài biến thể 1, 2 và 4, đối chiếu oracle:

In [28]:
def longest_at_most_k_types(arr, k):
    cnt, left, best = {}, 0, 0
    for right, x in enumerate(arr):
        cnt[x] = cnt.get(x, 0) + 1
        while len(cnt) > k:
            y = arr[left]
            cnt[y] -= 1
            if cnt[y] == 0:
                del cnt[y]
            left += 1
        best = max(best, right - left + 1)
    return best


def longest_at_most_k_brute(arr, k):
    n = len(arr)
    return max((j - i + 1 for i in range(n) for j in range(i, n) if len(set(arr[i:j + 1])) <= k), default=0)


def count_subarrays_at_most_k_types(arr, k):
    cnt, left, total = {}, 0, 0
    for right, x in enumerate(arr):
        cnt[x] = cnt.get(x, 0) + 1
        while len(cnt) > k:
            y = arr[left]
            cnt[y] -= 1
            if cnt[y] == 0:
                del cnt[y]
            left += 1
        total += right - left + 1                         # mọi đoạn kết thúc tại right bắt đầu từ left..right đều hợp lệ
    return total


def count_brute(arr, k):
    n = len(arr)
    return sum(1 for i in range(n) for j in range(i, n) if len(set(arr[i:j + 1])) <= k)


def best_window(arr):
    """Biến thể 4: trả (i, j) của một đoạn dài nhất có <= 2 loại (i nhỏ nhất)."""
    cnt, left, best, bi, bj = {}, 0, 0, 0, -1
    for right, x in enumerate(arr):
        cnt[x] = cnt.get(x, 0) + 1
        while len(cnt) > 2:
            y = arr[left]
            cnt[y] -= 1
            if cnt[y] == 0:
                del cnt[y]
            left += 1
        if right - left + 1 > best:
            best, bi, bj = right - left + 1, left, right
    return bi, bj


rng = random.Random(15)
for _ in range(500):
    a = [rng.randrange(5) for _ in range(rng.randint(1, 12))]
    for k in (1, 2, 3):
        assert longest_at_most_k_types(a, k) == longest_at_most_k_brute(a, k)
        assert count_subarrays_at_most_k_types(a, k) == count_brute(a, k)
    i, j = best_window(a)
    assert len(set(a[i:j + 1])) <= 2 and j - i + 1 == oracle(a)
print("k loại, đếm đoạn con, và trả đoạn tốt nhất: khớp vét cạn trên 500 mảng")
# Subarrays with exactly K different integers = at_most(K) - at_most(K - 1)
a = [1, 2, 1, 2, 3]
assert count_subarrays_at_most_k_types(a, 2) - count_subarrays_at_most_k_types(a, 1) == 7

k loại, đếm đoạn con, và trả đoạn tốt nhất: khớp vét cạn trên 500 mảng


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi điểm đầu | mở rộng tới loại thứ 3 | `O(n²)` | `O(1)` | oracle, `n` nhỏ |
| 2. Cửa sổ co bằng `while` | bộ đếm + co từ trái | `O(n)` | `O(1)` | **đáp án chính**, tổng quát `k` |
| 3. Cửa sổ chỉ trượt | độ dài không giảm | `O(n)` | `O(1)` | mã gọn |
| 4. Hai loại gần nhất | `a`, `b`, `run_b` | `O(n)` | `O(1)` | không `dict`, đặc thù 2 loại |

Cả bốn khớp oracle mọi-đoạn-con trên 87 380 mảng nhỏ và khớp nhau trên `n = 10^5` (mục 12).

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [29]:
print(Solution().totalFruit([3, 3, 3, 1, 2, 1, 1, 2, 3, 3, 4]), Solution().totalFruit([1, 1]), Solution().totalFruit([1, 2, 3, 4]), longest_at_most_k_types([1, 2, 3, 4], 3))

5 2 2 3


### Bài 2 (tính tay)

`fruits = [4, 4, 5, 6, 6, 5, 5, 4]`. (a) Điểm bắt đầu tốt nhất và đoạn tương ứng? (b) Đáp án? (c) Liệt kê `left` sau mỗi `right` khi chạy Cách 2. Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện cửa sổ co:

In [30]:
def window_todo(fruits):
    cnt, left, best = {}, 0, 0
    for right, x in enumerate(fruits):
        cnt[x] = cnt.get(x, 0) + 1
        while len(cnt) > 2:
            pass        # TODO 1: giảm đếm của fruits[left], xoá khoá nếu về 0, left += 1
        pass            # TODO 2: cập nhật best
    return best


# Khi làm xong, bỏ comment để kiểm tra:
# assert [window_todo(a) for a, _ in examples] == [3, 3, 4]

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **hai loại gần nhất** (không dùng `dict`):

In [31]:
class MySolution:
    def totalFruit(self, fruits):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().totalFruit(a) == w for a, w in examples)

### Bài 5 (mini challenge): Longest Substring with At Most K Distinct Characters (340)

Cho chuỗi `s` và `k`, tìm độ dài lớn nhất của chuỗi con có tối đa `k` ký tự khác nhau (`k` có thể bằng `0`). Viết `length_of_longest_substring_k_distinct(s, k)` và kiểm tra bằng vét cạn.

In [32]:
def length_of_longest_substring_k_distinct(s, k):
    pass            # TODO


# Khi làm xong:
# assert length_of_longest_substring_k_distinct("eceba", 2) == 3 and length_of_longest_substring_k_distinct("aa", 1) == 2

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: tìm đoạn dài nhất có ≤ 2 giá trị khác nhau.
- Mức 2: `[3,3,3,1,2,1,1,2,3,3,4]`: đoạn `[1,2,1,1,2]` dài `5`; `[2,3,3]` chỉ `3`.
- Mức 3: `5 2 2 3`: `[1,1]` → `2`; `[1,2,3,4]` → `2`; với `k = 3` → `3`.

**Bài 2**

- Mức 1: đoạn `[5,6,6,5,5]` dài `5` (loại `5` và `6`).
- Mức 2: bắt đầu tại `i = 2` tới `j = 6`; `[4,4,5]` chỉ `3`, còn thêm cây cuối loại `4` làm cửa sổ phải co mạnh.
- Mức 3: `left` sau mỗi `right`: `0,0,0,2,2,2,2,5`; đáp án `5` (đoạn `[5,6,6,5,5]`).

**Bài 3**

- Mức 1: `y = fruits[left]; cnt[y] -= 1`.
- Mức 2: `if cnt[y] == 0: del cnt[y]` và `left += 1`.
- Mức 3: `best = max(best, right - left + 1)` sau vòng `while`.

**Bài 4**

- Mức 1: biến `a, b, run_b, cur, best`.
- Mức 2: loại mới: `cur = run_b + 1`.
- Mức 3: cập nhật `(a, b, run_b)` như Cách 4.

**Bài 5**

- Mức 1: tổng quát hoá `2` thành `k`.
- Mức 2: `k = 0` trả `0`.
- Mức 3: `while len(cnt) > k`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [33]:
assert (Solution().totalFruit([3, 3, 3, 1, 2, 1, 1, 2, 3, 3, 4]), Solution().totalFruit([1, 1]), Solution().totalFruit([1, 2, 3, 4]), longest_at_most_k_types([1, 2, 3, 4], 3)) == (5, 2, 2, 3)
assert oracle([3, 3, 3, 1, 2, 1, 1, 2, 3, 3, 4]) == 5
print("Bài 1 OK: 5 2 2 3")

Bài 1 OK: 5 2 2 3


### Bài 2

In [34]:
a = [4, 4, 5, 6, 6, 5, 5, 4]
lefts = []
cnt, left = {}, 0
for right, x in enumerate(a):
    cnt[x] = cnt.get(x, 0) + 1
    while len(cnt) > 2:
        y = a[left]
        cnt[y] -= 1
        if cnt[y] == 0:
            del cnt[y]
        left += 1
    lefts.append(left)
assert lefts == [0, 0, 0, 2, 2, 2, 2, 5]
assert Solution().totalFruit(a) == oracle(a) == 5 and best_window(a) == (2, 6)
print("Bài 2 OK: left sau mỗi right =", lefts, "| đáp án", oracle(a), "| đoạn", best_window(a))

Bài 2 OK: left sau mỗi right = [0, 0, 0, 2, 2, 2, 2, 5] | đáp án 5 | đoạn (2, 6)


### Bài 3

In [35]:
def window_answer(fruits):
    cnt, left, best = {}, 0, 0
    for right, x in enumerate(fruits):
        cnt[x] = cnt.get(x, 0) + 1
        while len(cnt) > 2:
            y = fruits[left]
            cnt[y] -= 1
            if cnt[y] == 0:
                del cnt[y]
            left += 1
        best = max(best, right - left + 1)
    return best


assert [window_answer(a) for a, _ in examples] == [3, 3, 4]
for a in tests[:400]:
    assert window_answer(a) == oracle(a)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [36]:
class MySolutionAnswer:
    def totalFruit(self, fruits):
        a = b = -1
        run_b = cur = best = 0
        for x in fruits:
            cur = cur + 1 if (x == a or x == b) else run_b + 1
            if x == b:
                run_b += 1
            else:
                run_b = 1
                a, b = b, x
            best = max(best, cur)
        return best


assert all(MySolutionAnswer().totalFruit(a) == w for a, w in examples)
for a in tests[:400]:
    assert MySolutionAnswer().totalFruit(a) == oracle(a)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [37]:
def length_of_longest_substring_k_distinct_answer(s, k):
    if k == 0:
        return 0
    cnt, left, best = {}, 0, 0
    for right, ch in enumerate(s):
        cnt[ch] = cnt.get(ch, 0) + 1
        while len(cnt) > k:
            y = s[left]
            cnt[y] -= 1
            if cnt[y] == 0:
                del cnt[y]
            left += 1
        best = max(best, right - left + 1)
    return best


def k_distinct_brute(s, k):
    n = len(s)
    return max((j - i + 1 for i in range(n) for j in range(i, n) if len(set(s[i:j + 1])) <= k), default=0)


assert length_of_longest_substring_k_distinct_answer("eceba", 2) == 3 and length_of_longest_substring_k_distinct_answer("aa", 1) == 2
rng = random.Random(5)
for _ in range(400):
    s = "".join(rng.choice("abcd") for _ in range(rng.randint(1, 12)))
    k = rng.randint(0, 4)
    assert length_of_longest_substring_k_distinct_answer(s, k) == k_distinct_brute(s, k)
print("Bài 5 OK: khớp vét cạn")

Bài 5 OK: khớp vét cạn


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (đoạn con dài nhất với giới hạn số loại → cửa sổ trượt + bộ đếm):**

```python
class Solution:
    def totalFruit(self, fruits: List[int]) -> int:
        cnt, left, best = {}, 0, 0
        for right, x in enumerate(fruits):
            cnt[x] = cnt.get(x, 0) + 1
            while len(cnt) > 2:                 # quá 2 loại: co từ trái
                y = fruits[left]
                cnt[y] -= 1
                if cnt[y] == 0:
                    del cnt[y]                  # xoá khoá khi về 0
                left += 1
            best = max(best, right - left + 1)
        return best
```

**Checklist trước khi submit:**

- [ ] Hiểu: đáp án = đoạn con **liên tiếp** dài nhất có `≤ 2` loại?
- [ ] Xoá khoá khỏi `cnt` khi đếm về `0` (để `len(cnt)` đúng)?
- [ ] Co bằng `while` (không `if`); cập nhật `best` **sau** khi co?
- [ ] Độ dài `right − left + 1`?
- [ ] Đã thử `[1,2,1] → 3`, `[0,1,2,2] → 3`, `[1,2,3,2,2] → 4`, một cây, toàn một loại, mọi loại khác nhau?
- [ ] Đúng tên `class Solution`, `totalFruit(self, fruits)`, không in debug.